In [2]:
import pandas as pd
import uuid
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()



base_folder = BASE_DIR                  # already in uaf_neo4j_export_runner
input_folder = base_folder / "input"
output_folder = base_folder / "output"

input_folder.mkdir(exist_ok=True)
output_folder.mkdir(exist_ok=True)


In [3]:
print("Input folder absolute path:")
print(input_folder.resolve())
print("\nContents:")

if not input_folder.exists():
    print("❌ input folder does NOT exist")
else:
    for item in input_folder.iterdir():
        print(" -", item.name)




Input folder absolute path:
F:\OneDrive\_VSCode\UAF-Repo\UAF_to_NEO4J\input

Contents:
 - relationships.csv


In [4]:
csv_path = input_folder / "relationships.csv"
relationships = pd.read_csv(csv_path)


In [5]:
source_names = relationships["Source"]
target_names = relationships["Target"]

all_names = pd.concat([source_names, target_names]).drop_duplicates()

nodes = pd.DataFrame ({
    "id":["node_" + str(i) for i in range(1, len(all_names) +1)],
    "name": all_names
})

nodes

,id,name
0,node_1,Integrated Supply Chain Planning
1,node_2,Demand Planning
2,node_3,Anaplan
3,node_4,SAP S4
2,node_5,Demand Forecasting
3,node_6,Material Master


In [6]:
name_to_id = dict(zip(nodes["name"], nodes["id"]))

name_to_id

{'Integrated Supply Chain Planning': 'node_1',
 'Demand Planning': 'node_2',
 'Anaplan': 'node_3',
 'SAP S4': 'node_4',
 'Demand Forecasting': 'node_5',
 'Material Master': 'node_6'}

In [7]:
neo4j_relationships = []

for index, row in relationships.iterrows():
    source_name = row["Source"]
    target_name = row["Target"]
    relationship_type = row["Relationship"].upper().replace(" ", "_")

    neo4j_relationships.append({
        "id": str(uuid.uuid4()),
        "startId": name_to_id[source_name],
        "endId": name_to_id[target_name],
        "type": relationship_type
    })

neo4j_relationships = pd.DataFrame(neo4j_relationships)

neo4j_relationships

,id,startId,endId,type
0,d2e6fc80-1f70-4b17-990d-14bca0ea4d65,node_1,node_2,ENABLES
1,cc7441a3-5866-40bb-b281-b0d45cdfb493,node_2,node_3,USES
2,fc987a01-f6c1-498a-afa9-e7940b183ac6,node_3,node_5,MANAGES
3,46aba3fe-4429-49f2-af61-8eedb4dde1dd,node_4,node_6,MANAGES


In [8]:
nodes.to_csv(output_folder / "nodes.csv", index=False)
neo4j_relationships.to_csv(output_folder / "relationships.csv", index=False)

print("Created nodes.csv")
print("Created relationship.csv")

Created nodes.csv
Created relationship.csv
